<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Prompt Engineering für Agenten
</b></font> </br></p>

---

In [ ]:
#@title  🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "M04-Prompt-Engineering"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---



In diesem Modul lernen Sie, wie Prompts das Verhalten von Agenten steuern.

**Lernziele:**
- `ChatPromptTemplate` strukturiert aufbauen
- saubere `system`-Instruktionen für Tool-Calling schreiben
- Few-Shot-Beispiele gezielt einsetzen
- Prompt-Varianten vergleichen und bewerten

**Merksatz:** Bei Agenten ist Prompting nicht "nice to have", sondern Teil der Steuerlogik.


In [ ]:
# Imports für das Modul
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# 2 | ChatPromptTemplate
---

`ChatPromptTemplate` trennt klar zwischen Rollen (`system`, `human`, `ai`) und Variablen.
Das ist robuster als Prompt-Strings zusammenzukleben.


In [ ]:
prompt = load_prompt("../05_prompt/m04_python_tutor_prompt.md", mode="T")

messages = prompt.invoke({
    "thema": "for-Schleifen",
    "zielgruppe": "Einsteiger"
})

for msg in messages.messages:
    print(f"{msg.type.upper()}: {msg.content}")


In [ ]:
chain = prompt | llm
response = chain.invoke({"thema": "List Comprehension", "zielgruppe": "Data-Analysten"})
mprint(response.content)

# 3 | System Prompts für Tool-Calling
---



Für Agenten mit Tools sollte der `system`-Prompt explizit festlegen:
- wann ein Tool verwendet wird
- wann **kein** Tool verwendet wird
- welches Ausgabeformat erwartet wird


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Addiert zwei Zahlen und gibt das Ergebnis zurueck."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multipliziert zwei Zahlen und gibt das Ergebnis zurueck."""
    return a * b

system_prompt = load_prompt("../05_prompt/m04_rechenassistent_system_prompt.md", mode="T")

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[add, multiply],
    system_prompt=system_prompt,
)

result = agent.invoke({"messages": [{"role": "user", "content": "(12 + 8) * 3"}]})
mprint(result["messages"][-1].content)


# 4 | Few-Shot Examples
---



Few-Shot-Beispiele helfen, Stil und Struktur der Antworten zu stabilisieren.
Nutzen Sie wenige, aber repräsentative Beispiele.


In [ ]:
zero_shot = load_prompt("../05_prompt/m04_ticket_zero_shot_prompt.md", mode="T")
few_shot = load_prompt("../05_prompt/m04_ticket_few_shot_prompt.md", mode="T")
for name, p in [("Zero-Shot", zero_shot), ("Few-Shot", few_shot)]:
    # 1. Tracing-Konfiguration vorab festlegen
    run_cfg = {
        "run_name": f"M04_Kap4_{name.replace('-', '')}",
        "tags": ["M04", "few-shot", name.lower().replace("-", "")]
    }
    # 2. with_config() anwenden, dann invoke()
    out = (p | llm).with_config(**run_cfg).invoke({"ticket": "Wo finde ich meine letzte Rechnung?"})
    print()
    print(f"{name}: {out.content.strip()}")


# 5 | Prompt-Varianten testen
---



Ein einfacher Vergleich hilft, bessere Prompts datenbasiert auszuwählen.
Unten testen wir mehrere System-Prompts auf dieselbe Anfrage.


In [ ]:
variants = {
    "knapp": "Antworte in maximal 2 Saetzen.",
    "didaktisch": "Erklaere schrittweise mit kurzem Beispiel.",
    "kritisch": "Zeige Antwort und nenne eine moegliche Fehlerquelle.",
}
variant_prompt = load_prompt("../05_prompt/m04_varianten_prompt.md", mode="T")
user_question = "Wann sollte ich in Python eine Liste statt eines Sets verwenden?"
for name, instruction in variants.items():
    # 1. Tracing-Konfiguration vorab festlegen
    run_cfg = {
        "run_name": f"M04_Kap5_Variant_{name}",
        "tags": ["M04", "prompt-variants", name]
    }
    # 2. with_config() anwenden, dann invoke()
    answer = (variant_prompt | llm).with_config(**run_cfg).invoke({"instruction": instruction, "q": user_question})
    mprint("### Variante: " + name)
    mprint('---')
    mprint(answer.content.strip())


In [ ]:
# LangSmith: Aktives Projekt für diesen Abschnitt
import os
print(f"📊 LangSmith-Projekt: {os.environ['LANGCHAIN_PROJECT']}")

# 6 | LangSmith: Prompt-Inspektion
---



LangSmith macht Prompt-Experimente nachvollziehbar: Welcher Prompt erzeugte welche Antwort?

**Empfehlung im Kursbetrieb:**
- `run_name` pro Experiment setzen
- `tags` für Modul/Kapitel verwenden
- bei Vergleichen immer denselben Input nutzen

**Hinweis (EU):** Endpoint `https://eu.api.smith.langchain.com` – bereits in der Setup-Cell konfiguriert.

In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M04_Kap6_PromptBaseline",
    "tags":     ["M04", "prompt-engineering"]
}

# 2. with_config() anwenden, dann invoke()
traced_chain = (prompt | llm).with_config(**run_cfg)
traced_response = traced_chain.invoke({
    "thema": "while-Schleifen",
    "zielgruppe": "Python-Einsteiger"
})

mprint("### Trace erstellt")
mprint('---')
mprint(traced_response.content)

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

1. Erstellen Sie einen Agenten mit 2-3 Tools (z. B. Rechnen, Datum, String-Utils).
2. Schreiben Sie zwei unterschiedliche System-Prompts: einmal "knapp", einmal "didaktisch".
3. Definieren Sie 3 Testfragen und vergleichen Sie beide Varianten.
4. Dokumentieren Sie kurz: Welche Prompt-Version ist für Ihren Use Case besser und warum?

